# Giai doan 0 — ha tang

Chay tung o theo thu tu. Moi buoc tu bao `PASS` / `THIEU` / `HOI`.
Khong buoc nao crash vi thieu hien vat.

**Da chot** (ke-hoach §Da chot):
- retriever: `rmanluo/G-reasoner-34M`, checkpoint pretrain, **KHONG** chay `sft_training`
- corpus: train set HotpotQA + MuSiQue + 2Wiki — tieng Anh
- chunk: mot entry `documents.json` = mot document node = mot doan Wikipedia. **Khong cat lai.**

**Con treo**: model judge (#2) · model NLI (#3) · ba trich dan (#5) · bon cau hoi hien vat o `0.A`.

## 1 · Mount Drive va dat duong dan

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
ROOT = '/content/drive/MyDrive/ragproj'
STAGE0 = f'{ROOT}/stage0'
pathlib.Path(STAGE0).mkdir(parents=True, exist_ok=True)
os.chdir(STAGE0)
print(os.getcwd())
print(sorted(os.listdir('.')))

Neu thu muc trong: giai nen `stage0.zip` tu Drive vao day truoc.

```python
!unzip -o /content/drive/MyDrive/ragproj/stage0.zip -d /content/drive/MyDrive/ragproj/
```

## 2 · Cai dat

In [ ]:
# toi thieu de 0.A / 0.B / 0.J chay
!pip -q install pyyaml tiktoken
# 0.C
!pip -q install bm25s sentence-transformers
# 0.G
!pip -q install transformers torch --upgrade
# 0.E judge (DeepSeek) + 0.F che cau hoi
!pip -q install openai

import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('gpu :', p.name, '%.1f GB' % (p.total_memory / 1e9))
    # Qwen3-8B bf16 can ~16 GB.  <16 GB thi phai dung quantize hoac model nho hon.
    if p.total_memory / 1e9 < 16:
        print('CANH BAO: <16 GB — Qwen3-8B bf16 KHONG vua.')
        print('  chon: Qwen/Qwen3-4B, hoac ban AWQ/GPTQ cua 8B')
else:
    print('KHONG co GPU -> Runtime > Change runtime type > GPU')

In [ ]:
# test logic thuan — khong can hien vat, chay duoc ngay.
# PHAI pass truoc khi tin bat ky con so nao o duoi.
!python test_logic.py

## 3 · Lay hien vat

Tra loi cau hoi `0.A-1`: do thi da phat hanh san, hay phai tu dung?

In [ ]:
import pathlib
pathlib.Path('artifacts').mkdir(exist_ok=True)

# code
if not pathlib.Path('artifacts/gfm-rag').exists():
    !git clone --depth 1 https://github.com/RManLuo/gfm-rag artifacts/gfm-rag

# checkpoint (34M — nho)
!pip -q install huggingface_hub
from huggingface_hub import snapshot_download
ck = snapshot_download('rmanluo/G-reasoner-34M',
                       local_dir='artifacts/G-reasoner-34M')
print(ck)
print(sorted(p.name for p in pathlib.Path(ck).rglob('*') if p.is_file())[:40])

In [ ]:
# Tim xem tac gia co phat hanh do thi / dataset da xu ly khong.
# Neu KHONG co, 0.A se bao THIEU va ta phai dinh pham vi job tu dung.
from huggingface_hub import HfApi
api = HfApi()
for kind, it in (('model', api.list_models(author='rmanluo')),
                 ('dataset', api.list_datasets(author='rmanluo'))):
    for x in it:
        print(kind, '-', x.id)

### Model sinh — chay LOCAL trên GPU

`Qwen 3.5-9B` **không tồn tại**. Qwen3 (4/2025) dense có `0.6B · 1.7B · 4B · 8B · 14B · 32B` — không có 9B, không có dòng 3.5.
Gần nhất với "9B" là **`Qwen/Qwen3-8B`** (128K context).

Sinh chạy local nên **không tốn tiền, tốn GIỜ GPU** — và vì vậy `cache/judge` là bắt buộc để chạy lại được qua nhiều session Colab.

In [ ]:
# vLLM: nhanh hon transformers nhieu, va co prefix caching —
# dung thu ma vong 1 greedy can (moi ung vien la S + {k} voi cung S).
!pip -q install vllm

from huggingface_hub import snapshot_download
gen = snapshot_download('Qwen/Qwen3-8B', local_dir='artifacts/Qwen3-8B')
print(gen)

## 4 · Cau hinh

In [ ]:
import yaml, os, pathlib
cfg = yaml.safe_load(open('config.yaml', encoding='utf-8'))
cfg['paths']['root'] = '.'

# ── model sinh: LOCAL ────────────────────────────────────────────
# dung duong dan da tai, de khong tai lai moi session
cfg['gen']['model'] = 'artifacts/Qwen3-8B'
# GPU < 16 GB thi doi sang Qwen/Qwen3-4B

# ── model judge: DeepSeek qua API ───────────────────────────────
# PHAI la deepseek-chat.  deepseek-reasoner (thinking mode) NHAN tham so
# logprobs nhung KHONG CO TAC DUNG — loi IM LANG, 0.E se canh bao.
cfg['judge']['judge_model'] = 'deepseek-chat'

from google.colab import userdata
os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')

yaml.safe_dump(cfg, open('config.yaml', 'w', encoding='utf-8'),
               allow_unicode=True, sort_keys=False)
print('sinh  =', cfg['gen']['model'], '(local)')
print('judge =', cfg['judge']['judge_model'], '(DeepSeek API)')
print('key   =', 'co' if os.environ.get('DEEPSEEK_API_KEY') else 'CHUA CO')

## 5 · Chay tung buoc

`0.A` va `0.B` chay duoc ngay khi co do thi. `0.E` va `0.F` can quyet dinh #2.

In [ ]:
!python run_stage0.py 0.A

In [ ]:
# 0.B — do dai doan va chong lap.  Day la buoc RE NHAT va NGUY HIEM NHAT:
# 250 token/doan la gia dinh, va neu sai thi ca $12.535 sai theo,
# ma khong cong nao bat duoc — vi moi cong deu so voi chinh cong thuc do.
!python run_stage0.py 0.B

In [ ]:
!python run_stage0.py 0.G

In [ ]:
!python run_stage0.py 0.J

In [ ]:
# can bm25s + sentence-transformers, va GFM neu muon nhanh gfm
!python run_stage0.py 0.C

In [ ]:
# 0.E nap Qwen3-8B len GPU (lan dau mat vai phut) roi goi DeepSeek lam judge.
# No kiem BA thu:
#   1  v co LIEN TUC khong (dem so gia tri phan biet)
#   2  logprobs co THAT khong — DeepSeek V3.2 co bao cao tra toan 0/-9999,
#      va loi do IM LANG neu khong kiem gia tri
#   3  doan BAY ("hien tuong van hoa") co lam v cao gia khong -> ca moi tham so
!python run_stage0.py 0.E 0.F

## 6 · Tong hop

In [ ]:
!python run_stage0.py

In [ ]:
import json, pathlib
for p in sorted(pathlib.Path('reports').glob('*.json')):
    print('\n' + '=' * 70); print(p.name)
    d = json.loads(p.read_text(encoding='utf-8'))
    print(json.dumps(d, ensure_ascii=False, indent=2)[:2200])

## 7 · Bao cao lai

Gui lai `reports/stage0_summary.json`. Nhung con so quyet dinh:

| con so | buoc | vi sao quan trong |
|---|---|---|
| `token_moi_doan_p50/mean` | 0.B | thay `250` gia dinh; sai thi moi bang chi phi sai |
| `chong_lap` | 0.B | khac 0 thi phai khu trung lap TRUOC khi lap pool |
| `prompt_caching_an` | 0.E | sai thi phan tiet kiem nho tien to chung khong co |
| `che_cau_hoi_khop` | 0.F | thap thi bang phu thieu cot, T duoc nhan khi con thieu bang chung |
| `J_phan_bo` | 0.F | J la PRIOR cho `\|T\|` |
| `split_hieu_chuan` | 0.J | PHAI khong phai train, khong thi ca ba con so GD 1 phong gia |
| `chia_tap` | 0.J | holdout la tap duy nhat chua bi dao nhan |
| `covers_smaller_than_J` | 0.G | co nghia summary bao trum -> can quyet dinh §3.8 parsimony |